# Sentence Collection

In [64]:
import os
import re
import glob
import pandas as pd
from bs4 import BeautifulSoup  # for parsing XML/SGML
import spacy
from collections import Counter
import string

In [65]:
def extract_text(file_path):
    """Extract text from an XML/SGML file using BeautifulSoup."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            soup = BeautifulSoup(content, "html.parser")  # works for plain XML text
            return " ".join(soup.stripped_strings)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

def get_doc_id(filename, doc_type=None):
    """
    Extracts the document-level ID from the filename.
    Examples:
        OQ: 14011872__OQ__O-2007-0336__EN.txt -> O-2007-0336
        REPORT: 12345678__REPORT__A7-2010-0236__EN.txt -> A7-2010-0236
                 87654321__REPORT__A6-2009-0112__EN.txt -> A6-2009-0112
    """
    if doc_type == "REPORT":
        pattern = r'A\d+-\d{4}-\d+'
    elif doc_type == "OQ":
        pattern = r'O-\d{4}-\d+'
    else:
        # fallback: try both patterns
        pattern = r'(?:A\d+|O)-\d{4}-\d+'

    match = re.search(pattern, filename)
    return match.group(0) if match else None

In [83]:
languages = ["EN", "ES", "PL", "EL", "MT", "HU"]
pivot = "EN"
#pairs = [("EN", "RO"), ("EN", "LV")]
pairs = [("EN", lang) for lang in languages if lang != "EN"]

In [82]:
languages

['EN', 'ES', 'PL', 'EL', 'MT', 'HU']

In [84]:
import os
import glob
import pandas as pd

# ------------------- USER SETTINGS -------------------
BASE_PATH = "./"          # Base path to Lab1
SOURCE_FORMAT = "xml"    # "sgml" or "xml"
DOC_TYPE = "REPORT"           # Document type folder (e.g., REPORT, OQ, WQA)
# -----------------------------------------------------

# Container for all documents
all_documents = []

# Loop over languages
for lang in languages:
    sentence_path = os.path.join(BASE_PATH, lang, "sentence", SOURCE_FORMAT, lang, DOC_TYPE)
    source_path   = os.path.join(BASE_PATH, lang, "source", SOURCE_FORMAT, lang, DOC_TYPE)

    if os.path.exists(sentence_path):
        doc_path = sentence_path
    elif os.path.exists(source_path):
        doc_path = source_path
    else:
        print(f"⚠️ No valid folder found for {lang}")
        continue

    files = glob.glob(os.path.join(doc_path, "*"))
    files = [f for f in files if f.lower().endswith((".txt", ".xml", ".sgml"))]

    for file in files:
        text = extract_text(file)
        filename = os.path.basename(file)

        # Extract text_id (everything before last __LANG)
        text_id = os.path.splitext(filename)[0].rsplit("__", 1)[0]

        # Extract document-level ID from filename
        doc_id = get_doc_id(filename)

        all_documents.append({
            "text_id": text_id,
            "doc_id": doc_id,
            "filename": filename,
            "language": lang,
            "category": DOC_TYPE,
            "source_format": SOURCE_FORMAT,
            "text": text
        })

# Convert to DataFrame
corpus_df = pd.DataFrame(all_documents)

# Quick check
print(corpus_df.head())
print(f"Total documents collected: {len(corpus_df)}")

print(languages)

for lang, count in corpus_df["language"].value_counts().items():
    print(f"{lang}: {count} documents")

                          text_id        doc_id  \
0  26645894__REPORT__A7-2010-0236  A7-2010-0236   
1  24919351__REPORT__A7-2010-0041  A7-2010-0041   
2  20479401__REPORT__A6-2008-0435  A6-2008-0435   
3  11802250__REPORT__A6-2005-0369  A6-2005-0369   
4  25097283__REPORT__A7-2010-0099  A7-2010-0099   

                                 filename language category source_format  \
0  26645894__REPORT__A7-2010-0236__EN.txt       EN   REPORT           xml   
1  24919351__REPORT__A7-2010-0041__EN.txt       EN   REPORT           xml   
2  20479401__REPORT__A6-2008-0435__EN.txt       EN   REPORT           xml   
3  11802250__REPORT__A6-2005-0369__EN.txt       EN   REPORT           xml   
4  25097283__REPORT__A7-2010-0099__EN.txt       EN   REPORT           xml   

                                                text  
0  A7-0236/2010\n26.7.2010\nREPORT\non the fundin...  
1  A7-0041/2010\n17.3.2010\nREPORT\non the nomina...  
2  A6-0435/2008\n11.11.2008\nREPORT\non the situa...  
3  FINAL\n

## Parallel Filtering

In [85]:
# Step 1: Group by language and get sets of doc_ids
doc_ids_per_lang = corpus_df.groupby('language')['doc_id'].unique()
sets = [set(doc_ids_per_lang[lang]) for lang in languages]

# Step 2: Intersection to find common documents
common_doc_ids = set.intersection(*sets)
print(f"Documents present in all languages: {len(common_doc_ids)}")

# Step 3: Filter corpus to only include common doc_ids
parallel_corpus_df = corpus_df[corpus_df['doc_id'].isin(common_doc_ids)].reset_index(drop=True)

# Quick check
print(parallel_corpus_df.head())
print(f"Total documents after filtering: {len(parallel_corpus_df)}")

Documents present in all languages: 2698
                          text_id        doc_id  \
0  26645894__REPORT__A7-2010-0236  A7-2010-0236   
1  24919351__REPORT__A7-2010-0041  A7-2010-0041   
2  20479401__REPORT__A6-2008-0435  A6-2008-0435   
3  11802250__REPORT__A6-2005-0369  A6-2005-0369   
4  25097283__REPORT__A7-2010-0099  A7-2010-0099   

                                 filename language category source_format  \
0  26645894__REPORT__A7-2010-0236__EN.txt       EN   REPORT           xml   
1  24919351__REPORT__A7-2010-0041__EN.txt       EN   REPORT           xml   
2  20479401__REPORT__A6-2008-0435__EN.txt       EN   REPORT           xml   
3  11802250__REPORT__A6-2005-0369__EN.txt       EN   REPORT           xml   
4  25097283__REPORT__A7-2010-0099__EN.txt       EN   REPORT           xml   

                                                text  
0  A7-0236/2010\n26.7.2010\nREPORT\non the fundin...  
1  A7-0041/2010\n17.3.2010\nREPORT\non the nomina...  
2  A6-0435/2008\n11.11.2

In [36]:
parallel_corpus_df[['language', 'text']].to_csv('corpus_for_zipf.csv', index=False)

## Text Preprocessing

In [88]:
import spacy
from num2words import num2words
import dateparser

nlp = spacy.load("en_core_web_sm")  # English; swap for other languages if needed

def smart_tokenize(text):
    if not isinstance(text, str):
        return []
    
    doc = nlp(text)
    tokens = []

    for ent in doc.ents:
        # Replace multi-word named entities with a single token
        text = text.replace(ent.text, ent.text.replace(" ", "_"))

    doc = nlp(text)  # reprocess text after replacing multi-word proper nouns

    for token in doc:
        tok = token.text

        # 1. Handle numbers
        if tok.isdigit():
            tok = num2words(int(tok))

        # 2. Handle dates
        # Try parsing tokens that look like dates
        elif "." in tok or "/" in tok:
            dt = dateparser.parse(tok, settings={'DATE_ORDER': 'DMY'})
            if dt:
                tok = dt.strftime("%B_%d_%Y")  # e.g., 17.03.2002 -> March_17_2002

        tokens.append(tok)

    return tokens

In [89]:
parallel_corpus_df['text_clean'] = parallel_corpus_df['text'].apply(smart_tokenize)

In [ ]:
# Quick check: print first text for each language
for lang in parallel_corpus_df['language'].unique():
    example = parallel_corpus_df[parallel_corpus_df['language'] == lang].iloc[0]
    print(f"--- {lang} example ---")
    print(example['text_clean'][:1000])
    print("\n")

In [55]:
# Example: remove all digits from text
def remove_numbers(text):
    if isinstance(text, str):
        # replace digits with empty string
        return re.sub(r'\d+', '', text)
    return text

def remove_punctuation(text):
    return re.sub(rf"[{re.escape(string.punctuation)}]", "", text)    

def remove_long_urls(text, max_length=50):
    if not isinstance(text, str):
        return text
    words = text.split()
    cleaned_words = [
        w for w in words 
        if len(w) <= max_length and not re.match(r'^(http|www|[a-z]+org|[a-z]+eu)', w)
    ]
    return ' '.join(cleaned_words)  


# List of EU (and nearby) countries you want to remove
countries_to_remove = [
    'belarus', 'moldova', 'ukraine', 'armenia', 'azerbaijan', 'georgia', 
    'germany', 'france', 'italy', 'spain', 'poland', 'netherlands', 
    'belgium', 'luxembourg', 'latvia', 'lithuania', 'estonia', 
    'czechia', 'slovakia', 'hungary', 'romania', 'bulgaria', 'croatia', 
    'slovenia', 'greece', 'portugal', 'malta', 'finland', 'sweden', 
    'denmark', 'ireland', 'cyprus'
]

def remove_countries(text):
    if not isinstance(text, str):
        return text
    for country in countries_to_remove:
        # replace country name with a space instead of empty string
        text = re.sub(country, ' ', text, flags=re.IGNORECASE)
    return text

# Apply to your parallel_corpus_df
parallel_corpus_df['text_clean'] = parallel_corpus_df['text'].apply(remove_numbers)
parallel_corpus_df['text_clean'] = parallel_corpus_df['text_clean'].apply(remove_punctuation)  # first remove punctuation
parallel_corpus_df['text_clean'] = parallel_corpus_df['text_clean'].apply(remove_countries)    # then remove countries
parallel_corpus_df['text_clean'] = parallel_corpus_df['text_clean'].apply(remove_long_urls)    # then long URLs
parallel_corpus_df['text_clean'] = parallel_corpus_df['text_clean'].str.lower()
parallel_corpus_df['text_clean'] = parallel_corpus_df['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Quick check: print first text for each language
for lang in parallel_corpus_df['language'].unique():
    example = parallel_corpus_df[parallel_corpus_df['language'] == lang].iloc[0]
    print(f"--- {lang} example ---")
    print(example['text_clean'][:1000])
    print("\n")

--- EN example ---
a report on the funding and functioning of the european globalisation adjustment fund ini committee on budgets rapporteur miguel portas rapporteur for the opinion elisabeth morinchartier committee on employment and social affairs associated committees rule of the rules of procedure pe v contents motion for a european parliament resolution explanatory statement opinion of the committee on employment and social affairs opinion of the committee on economic and monetary affairs result of final vote in committee associated committees rule of the rules of procedure motion for a european parliament resolution on the funding and functioning of the european globalisation adjustment fund ini the european parliament – having regard to the interinstitutional agreement of may between the european parliament the council and the commission on budgetary discipline and sound financial management oj c p iia of may and in particular point thereof – having regard to regulation ec no of 

In [56]:
parallel_corpus_df[['language', 'text_clean']].to_csv('corpus_for_zipf.csv', index=False)

# Trash

In [34]:

models = {
    "EN": spacy.load("en_core_web_sm"),
    "RO": spacy.load("ro_core_news_sm"),   # after downloading
    "LV": spacy.load("xx_ent_wiki_sm"),    # fallback multilingual
}

def remove_names(text, lang):
    nlp = models[lang]
    doc = nlp(text)
    tokens = [token.text for token in doc if token.ent_type_ != "PERSON"]
    return " ".join(tokens)

# Make a copy so we don't overwrite the original
filtered_corpus_df = parallel_corpus_df.copy()

# Apply remove_names to each row, using the language column
filtered_corpus_df['text'] = filtered_corpus_df.apply(
    lambda row: remove_names(row['text'], row['language']),
    axis=1
)

# Now filtered_corpus_df is still a DataFrame, with names removed
print(filtered_corpus_df.head())    

KeyboardInterrupt: 

In [ ]:
filtered_corpus_df.groupby('language')['text'].apply(lambda x: x.str.len().mean())

language
EN    2039.395813
LV    2114.967099
RO    2232.383848
Name: text, dtype: float64

In [35]:
# Load English spaCy model
nlp_en = spacy.load("en_core_web_sm")

# Dictionary to track removed names per document
removed_names_dict = {}

def remove_names_english(text, text_id):
    """
    Remove PERSON entities from English text using spaCy.
    Return cleaned text, list of names removed, and count.
    """
    doc = nlp_en(text)
    removed_names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    # Keep track
    removed_names_dict[text_id] = removed_names
    # Remove names from text
    cleaned_text = text
    for name in removed_names:
        cleaned_text = re.sub(r'\b{}\b'.format(re.escape(name)), '', cleaned_text)
    return cleaned_text, len(removed_names)

def remove_names_other_lang(text, text_id):
    """
    Remove names in other languages based on English removed names.
    Returns cleaned text and count of removed names.
    """
    names_to_remove = removed_names_dict.get(text_id, [])
    count_removed = 0
    cleaned_text = text
    for name in names_to_remove:
        matches = re.findall(r'\b{}\b'.format(re.escape(name)), cleaned_text)
        count_removed += len(matches)
        cleaned_text = re.sub(r'\b{}\b'.format(re.escape(name)), '', cleaned_text)
    return cleaned_text, count_removed

# Example: process parallel_corpus_df
name_counts = {"EN": [], "RO": [], "LV": []}
cleaned_texts = []

for idx, row in parallel_corpus_df.iterrows():
    text_id = row['text_id']
    lang = row['language']
    text = row['text']
    
    if lang == "EN":
        cleaned, n_removed = remove_names_english(text, text_id)
    else:
        cleaned, n_removed = remove_names_other_lang(text, text_id)
    
    name_counts[lang].append(n_removed)
    
    cleaned_texts.append({
        "text_id": text_id,
        "language": lang,
        "text": cleaned,
        "names_removed": n_removed
    })

# Convert to DataFrame
filtered_corpus_df = pd.DataFrame(cleaned_texts)

# Summary stats
for lang in ["EN", "RO", "LV"]:
    total_removed = sum(name_counts[lang])
    print(f"{lang}: {total_removed} names removed across {len(name_counts[lang])} documents (avg {total_removed/len(name_counts[lang]):.2f} per doc)")

KeyboardInterrupt: 

In [ ]:
# For each language, print the first text
for lang in ["EN", "RO", "LV"]:
    example = filtered_corpus_df[filtered_corpus_df['language'] == lang].iloc[0]  # first row for that language
    print(f"--- {lang} example ---")
    print(example['text'][:1000])  # print first 1000 characters to avoid huge output
    print("\n")

--- EN example ---
ORAL QUESTION WITH DEBATE O-0113/09 
 pursuant to Rule 115 of the Rules of Procedure 
 by 
 
 , on behalf of the ALDE Group 
 to the Commission 
 Subject : Meat imports from Brazil and other non - Community countries 
 The Food and Veterinary Office ( FVO ) carried out inspections in Brazil between 20 January and 2 February 2009 to check that Brazilian exports comply with Community standards . 
 From the report published subsequently by the FVO , it would seem that some certification bodies do not meet the standards required . 
 In view of the irregularities found by the FVO , can the Commission guarantee that the 1 500 Brazilian holdings authorised to export to the EU meet the Community ’s requirements ? 
 The FVO ’s report also revealed serious deficiencies in the traceability system run by the Brazilian holdings , especially with regard to animal identification and databases . 
 In light of the inspection results , what guarantees can European consumers be given t